[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/a1cd/Shears/blob/main/Notebooks/Shears.ipynb)

# Shears: prune and heal a Mixture-of-Experts model

> ## Read this first
>
> **The real target is MLX on an Apple Silicon Mac, with Gemma 4 26B-A4B.** Every number published for Shears (speed, memory, quality, energy) was measured there.
>
> **This notebook runs a tiny random model, so you can watch the whole pipeline work.** It is not a way to get a good pruned model. The tiny model has 4 layers and 8 experts per layer, and it writes gibberish before and after. On Colab it runs on the **CPU** backend of MLX, which is a different program in practice: no Metal, no zero-copy memory mapping, no unified memory, different numerics. Do not compare its timings or losses with the Mac numbers.
>
> **The CUDA path is untested.** It exists as one optional, off-by-default cell near the end of the setup. Nobody has run it.
>
> If something here fails or looks slow, that says nothing about Shears on a Mac.

Shears prunes a Mixture-of-Experts model (removes experts) and heals it (retrains what is left, with small adapters, against the original model). It is a Swift library on [mlx-swift](https://github.com/ml-explore/mlx-swift). Python is only glue in this notebook: it finds your platform, runs shell commands and, at the end, loads the result with Hugging Face transformers.

**What runs, in order**

1. Detect the platform (Apple Silicon Mac, or Linux such as Colab).
2. Set up `shears-cli`. On a Mac that is a build with `xcodebuild`. On x86_64 Linux (Colab) it downloads a prebuilt binary that CI builds from this repository and checks its SHA-256; nothing is built. (Building from source on Linux is an optional cell at the very end, for changing Shears itself.)
3. Run the pipeline on a tiny random Gemma 4 MoE: `select` (choose which experts to keep), `prepass` (run the original model once and store its outputs), `heal` (train the adapters, three stages), `export` (merge them, write only the kept experts, Hugging Face layout, bf16).
4. Load that export with transformers and generate a few tokens.

**Time.** On a Mac (M3 Max) a cold setup takes about 100 seconds once Xcode is ready, and the pipeline 2 to 15 seconds (the first run compiles Metal kernels). On Linux the setup is a download of a few tens of megabytes and takes seconds to a minute (CI measures it on every change, on a plain x86_64 Ubuntu 24.04 runner; **not yet measured on Colab itself**). Building the binary yourself instead (the optional cell at the end) takes on the order of 15 to 30 minutes on a Colab CPU runtime (an estimate from builds on a 4-core x86_64 machine; **not measured on Colab**). The pipeline itself then takes seconds. Section 5 prints the time of each phase.

**On Colab** use a CPU runtime (the GPU is not used unless you turn on the experimental cell). Run the cells top to bottom; each shell step runs in its own `bash`, with `PATH` set from Python.

## 1. Platform and settings

In [ ]:
import os, sys, platform, shutil, subprocess, time, json, pathlib

# --- Settings ---------------------------------------------------------------------------------
SHEARS_GIT_URL = os.environ.get("SHEARS_GIT_URL", "https://github.com/a1cd/Shears.git")   # used only when this notebook is not inside a Shears checkout
SHEARS_REF = os.environ.get("SHEARS_REF", "notebook")        # branch or tag to build in that case (change to "main" once this notebook is on main)
SWIFT_VERSION = "6.3.3"        # Linux only; the version the Linux build was verified with
TRANSFORMERS_VERSION = "5.20.2"  # pinned. Gemma 4 first appears in transformers 5.5.0; 5.20.2 is the version tested here
RUN_EXPERIMENTAL_CUDA = False  # Linux + NVIDIA GPU only. UNTESTED. Only used by the optional build-from-source cell at the end; read it before turning this on.
# Linux x86_64: the notebook downloads this prebuilt shears-cli (CPU) and checks its SHA-256, which is read from PREBUILT_URL + ".sha256"
# unless you pin it in PREBUILT_SHA256. A local file works too (a path or a file:// URL). If the download fails the notebook stops;
# the optional last cell builds from source instead.
PREBUILT_URL = os.environ.get("SHEARS_PREBUILT_URL", "https://github.com/a1cd/Shears/releases/download/linux-x86_64-latest/shears-cli-linux-x86_64.tar.gz")
PREBUILT_SHA256 = os.environ.get("SHEARS_PREBUILT_SHA256", "")

# --- Platform ---------------------------------------------------------------------------------
IS_MAC = sys.platform == "darwin"
IS_LINUX = sys.platform.startswith("linux")
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ
ARCH = platform.machine()
if IS_MAC and ARCH != "arm64":
    raise SystemExit("Shears needs Apple Silicon (MLX does not run on Intel Macs). Use Colab or Linux instead.")
if not (IS_MAC or IS_LINUX):
    raise SystemExit(f"Unsupported platform: {sys.platform}. Use an Apple Silicon Mac or x86_64 Linux.")
print(f"platform: {'Apple Silicon Mac' if IS_MAC else 'Linux'} ({ARCH}){', Colab' if IN_COLAB else ''}, Python {platform.python_version()}")
if IS_LINUX:
    print(open("/etc/os-release").read().split("PRETTY_NAME=")[1].splitlines()[0], "| CPUs:", os.cpu_count())

# --- Where things live ------------------------------------------------------------------------
# SwiftPM names a path dependency after its directory, so the sources must sit in a directory called "Shears".
WORK = pathlib.Path(os.environ.get("SHEARS_WORK") or ("/content/shears-work" if IN_COLAB else pathlib.Path.home() / "shears-notebook-work"))
SRC = WORK / "Shears"
WORK.mkdir(parents=True, exist_ok=True)

# Inside a checkout (Notebooks/ next to Package.swift)? Use that code. Otherwise clone it.
here = pathlib.Path.cwd()
LOCAL_REPO = next((p for p in (here.parent, here) if (p / "Package.swift").exists() and (p / "Sources" / "Shears").is_dir()), None)

# --- Helpers ----------------------------------------------------------------------------------
PHASES = []   # (name, seconds)

def sh(script, name, cwd=None, env=None, quiet=False):
    """Run a bash script (stops at the first error), stream its output, and record how long it took.
    quiet=True prints only the last few lines, or the last 60 if it fails (for noisy Xcode builds)."""
    e = dict(os.environ, **(env or {}))
    t0 = time.time()
    p = subprocess.Popen(["bash", "-c", "set -eo pipefail\n" + script], cwd=cwd, env=e,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in p.stdout:
        if quiet:
            tail = (tail + [line])[-60:]
        else:
            print(line, end="")
    rc = p.wait()
    dt = time.time() - t0
    PHASES.append((name, dt))
    if quiet:
        print("".join(tail if rc != 0 else []), end="")
    print(f"--- {name}: {dt:.1f} s")
    if rc != 0:
        raise RuntimeError(f"{name} failed with exit status {rc}")

def add_to_path(*dirs):
    os.environ["PATH"] = os.pathsep.join([str(d) for d in dirs] + [os.environ["PATH"]])

def phase_table():
    for n, s in PHASES:
        print(f"{n:42} {s:8.1f} s")
    print(f"{'total':42} {sum(s for _, s in PHASES):8.1f} s")

print("work directory:", WORK)
print("Shears code from:", LOCAL_REPO if LOCAL_REPO else f"{SHEARS_GIT_URL} @ {SHEARS_REF}")

## 2. Setup: get `shears-cli`

First the sources go into `SRC` (a copy of this checkout, or a fresh clone; the pipeline reads its example config). Then `shears-cli` itself: on a Mac it is built with `xcodebuild` (2a); on x86_64 Linux and Colab the prebuilt binary is downloaded (2b). The cell for the other platform prints that it is skipping.

In [ ]:
# Put the sources in WORK/Shears.
if LOCAL_REPO:
    if SRC.exists():
        shutil.rmtree(SRC)
    shutil.copytree(LOCAL_REPO, SRC, ignore=shutil.ignore_patterns(".git", ".build", "Packages", ".swiftpm", "DerivedData", "*.xcodeproj", "__pycache__", ".ipynb_checkpoints"))
    print("copied", LOCAL_REPO, "->", SRC)
elif not (SRC / "Package.swift").exists():
    sh(f'git clone --depth 1 --branch "{SHEARS_REF}" "{SHEARS_GIT_URL}" "{SRC}"', "git clone Shears")
sh("git log -1 --format='sources at %h %s' 2>/dev/null || true", "sources", cwd=SRC)

### 2a. Apple Silicon Mac

Needs Xcode with the Metal Toolchain (`xcodebuild -downloadComponent MetalToolchain` if the build complains about `metal`). Plain `swift build` cannot compile MLX's Metal shaders, so this uses `xcodebuild`.

In [ ]:
if IS_MAC:
    sh("xcrun -f metal >/dev/null 2>&1 || { echo 'No Metal Toolchain. Run: xcodebuild -downloadComponent MetalToolchain'; exit 1; }", "check Metal toolchain")
    sh(f'cd "{SRC}/CLI" && xcodebuild build -scheme shears-cli -destination "platform=macOS" -configuration Release '
       f'-derivedDataPath "{WORK}/derived" -quiet 2>&1', "xcodebuild shears-cli", quiet=True)
    SHEARS_CLI = WORK / "derived/Build/Products/Release/shears-cli"
else:
    print("not a Mac: skipping (the Linux cells below do the setup)")

### 2b. Linux x86_64 and Colab: the prebuilt `shears-cli`

On x86_64 Linux this cell downloads a prebuilt `shears-cli` (CPU, Swift runtime linked statically, the few shared libraries it needs bundled next to it) from `PREBUILT_URL`, checks its SHA-256, unpacks it, runs it once as a test, installs `g++` if it is missing (MLX compiles fused CPU kernels with it at run time) and puts the binary on `PATH`. CI builds it from this repository on Ubuntu 24.04; it needs glibc 2.39 or newer.

If any step fails (no network, no such file, wrong checksum, the binary does not run on this system) the cell stops and says why. Nothing is built behind your back: to build from source instead, use the optional cell at the end of the notebook, then rerun from section 3.


In [ ]:
PREBUILT_OK = False
if IS_LINUX and ARCH != "x86_64":
    raise RuntimeError(f"There is no prebuilt shears-cli for Linux {ARCH}. Build from source with the optional cell at the end of the notebook.")
if IS_LINUX:
    import hashlib, tarfile, urllib.request
    t0 = time.time()
    PRE = WORK / "prebuilt"
    try:
        def fetch(url, dest):
            if "://" not in url:
                url = pathlib.Path(url).resolve().as_uri()
            req = urllib.request.Request(url, headers={"User-Agent": "shears-notebook"})
            with urllib.request.urlopen(req, timeout=120) as r, open(dest, "wb") as f:
                shutil.copyfileobj(r, f)

        shutil.rmtree(PRE, ignore_errors=True)
        PRE.mkdir(parents=True)
        print("downloading", PREBUILT_URL)
        fetch(PREBUILT_URL, PRE / "shears-cli.tar.gz")
        want = PREBUILT_SHA256.strip().lower()
        if not want:
            fetch(PREBUILT_URL + ".sha256", PRE / "shears-cli.tar.gz.sha256")
            want = (PRE / "shears-cli.tar.gz.sha256").read_text().split()[0].lower()
        h = hashlib.sha256()
        with open(PRE / "shears-cli.tar.gz", "rb") as f:
            for block in iter(lambda: f.read(1 << 20), b""):
                h.update(block)
        if h.hexdigest() != want:
            raise ValueError(f"SHA-256 mismatch: expected {want}, got {h.hexdigest()}")
        print("SHA-256 ok:", want, f"({(PRE / 'shears-cli.tar.gz').stat().st_size / 1e6:.1f} MB)")
        with tarfile.open(PRE / "shears-cli.tar.gz") as tf:
            if hasattr(tarfile, "data_filter"):
                tf.extractall(PRE / "pkg", filter="data")
            else:
                tf.extractall(PRE / "pkg")
        cli = next((PRE / "pkg").glob("*/bin/shears-cli"))
        # Does this binary run on this system at all (glibc, libstdc++, the bundled libraries)?
        r = subprocess.run([str(cli), "--help"], capture_output=True, text=True, timeout=120)
        if r.returncode != 0:
            raise RuntimeError(f"shears-cli --help exited {r.returncode}: {(r.stderr or r.stdout).strip()[:500]}")
        PHASES.append(("prebuilt shears-cli (download, verify, unpack, test)", time.time() - t0))
        # g++ is needed at run time: MLX compiles fused CPU kernels with it.
        sh(r"""
SUDO=""; [ "$(id -u)" = 0 ] || SUDO=sudo
if ! command -v g++ >/dev/null; then
  export DEBIAN_FRONTEND=noninteractive
  $SUDO apt-get update -qq && $SUDO apt-get install -y -qq g++
fi
g++ --version | head -1
""", "g++ for MLX's CPU kernels")
        SHEARS_CLI = cli
        add_to_path(cli.parent)
        PREBUILT_OK = True
        print("using the prebuilt shears-cli:", cli)
    except Exception as e:
        PHASES.append(("prebuilt shears-cli (failed)", time.time() - t0))
        raise RuntimeError(f"""Could not set up the prebuilt shears-cli: {type(e).__name__}: {e}

Nothing was built. Check the network and PREBUILT_URL in section 1, then rerun this cell.
To build shears-cli from source instead (15 to 30 minutes), set BUILD_FROM_SOURCE = True in the optional cell at the end of the notebook, run it, and then rerun from section 3.""") from None
else:
    print("not Linux: skipping")


## 3. Prune and heal a tiny model

`shears-cli sample` writes a tiny random Gemma 4 MoE (`model/`: 4 layers, 8 experts each, a toy tokenizer, float32) and `rows.jsonl` (four short token rows to heal on) into a folder. `Examples/tiny-config.json` is the config for it; it names exactly those paths. The notebook cuts it to 3 heal steps per stage: this only shows the pipeline runs. Then the four commands. Each reads the same config and is safe to rerun: it resumes.

In [ ]:
SETUP_MODE = "xcodebuild" if IS_MAC else ("prebuilt" if PREBUILT_OK else "source")   # source: the optional cell at the end was run
SETUP_SECONDS = sum(s for _, s in PHASES)
print(f"setup: {SETUP_MODE}, {SETUP_SECONDS:.1f} s")
TINY = WORK / "tiny"
shutil.rmtree(TINY, ignore_errors=True)
sh(f'"{SHEARS_CLI}" sample "{TINY}"', "sample workspace")
# Trim the sample config to a smoke test: 3 heal steps per stage (the sample has 6, 6 and 5).
cfg = json.load(open(SRC / "Examples/tiny-config.json"))
for st in cfg["stages"]:
    st["steps"] = 3
cfg["stages"][2]["plan"] = "freerun@0,end@2"   # stage C still reaches its second phase
json.dump(cfg, open(TINY / "config.json", "w"), indent=2)
print(sorted(p.name for p in TINY.iterdir()))

**`select`** picks 4 of the 8 experts per layer to keep, by how much dropping the others changes the layer's output on the rows.

In [ ]:
sh(f'cd "{TINY}" && "{SHEARS_CLI}" select --config config.json', "shears-cli select")
for i, kept in enumerate(json.load(open(TINY / "out/keep.json"))["keep"]):
    print(f"layer {i}: keeps experts {kept} of 0-7")

**`prepass`** runs the original (unpruned) model once over the rows and stores the outputs it needs, so healing never has to run the big model again. This is the "teacher".

In [ ]:
sh(f'cd "{TINY}" && "{SHEARS_CLI}" prepass --config config.json', "shears-cli prepass")

**`heal`** trains the adapters (small add-on matrices per expert, for the dense layers, and a low-rank correction to the router) so the pruned model's outputs move back toward the teacher's. The config has three stages: A and B heal one block of layers at a time, C runs over the whole pruned model. The loss is printed per step. (A few toy steps on a random model say nothing about quality, and the loss need not fall: stage C switches between training phases (freerun, then end) that use different losses. This only shows the loop running. `mlx peak` reads 0.00 GB on the CPU backend.)

In [ ]:
sh(f'cd "{TINY}" && "{SHEARS_CLI}" heal --config config.json', "shears-cli heal")
for stage in "ABC":
    rows = [json.loads(l) for l in open(TINY / f"out/stage-{stage}/steps.jsonl")]
    print(f"stage {stage}: {len(rows)} steps, loss {rows[0]['loss']:.3f} -> {rows[-1]['loss']:.3f}")

**`export`** merges the healed adapters into the weights and writes only the kept experts, in Hugging Face layout (`config.json`, safetensors shards, tokenizer files), as bfloat16. The same directory feeds `shears-cli gguf` (a GGUF file for llama.cpp) and `shears-serve` (an OpenAI-style server).

In [ ]:
sh(f'cd "{TINY}" && "{SHEARS_CLI}" export --config config.json', "shears-cli export")
EXPORT = TINY / "out/export"
for p in sorted(EXPORT.iterdir()):
    print(f"{p.stat().st_size:>9,}  {p.name}")
cfg = json.load(open(EXPORT / "config.json"))
print("\nexperts per layer in the export:", cfg["num_experts"], "(the original had 8); experts used per token:", cfg["top_k_experts"])

## 4. Load the export with Hugging Face transformers

The export is an ordinary Hugging Face checkpoint, so `AutoModelForCausalLM` loads it (a text-only Gemma 4, `Gemma4ForCausalLM`). transformers is pinned to a version that has Gemma 4. torch is installed too if the environment has none (Colab has one).

If you are on Colab and `import transformers` ran earlier in this session, restart the runtime after this cell so the pinned version is the one imported.

In [ ]:
need = [f"transformers=={TRANSFORMERS_VERSION}"]
try:
    import torch
except ImportError:
    need.append("torch")
sh(f'"{sys.executable}" -m pip install -q --disable-pip-version-check {" ".join(need)}', "pip install transformers")
import transformers
print("transformers", transformers.__version__)
assert transformers.__version__ == TRANSFORMERS_VERSION, "restart the runtime so the pinned transformers is imported"

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
transformers.utils.logging.disable_progress_bar()

tokenizer = AutoTokenizer.from_pretrained(EXPORT)
model = AutoModelForCausalLM.from_pretrained(EXPORT, dtype=torch.bfloat16)   # the export is bf16
model.eval()
print(type(model).__name__, f"| {sum(p.numel() for p in model.parameters()):,} parameters | dtype {next(model.parameters()).dtype}")

# --- Change the prompt here. The tokenizer is a toy (lower-case letters and bytes), so use short plain text.
messages = [{"role": "user", "content": "hello"}]
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True)

# --- Change sampling here. A tiny RANDOM model writes gibberish whatever you do; this only shows the model runs.
# For varied output set do_sample=True and temperature / top_k / top_p. Greedy decoding is repeatable.
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=5, do_sample=False,
                         pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)
new_tokens = out[0, inputs["input_ids"].shape[1]:]
print("prompt   :", repr(tokenizer.decode(inputs["input_ids"][0])))
print("generated:", repr(tokenizer.decode(new_tokens)))
print("token ids:", new_tokens.tolist())

## 5. How long each phase took

In [ ]:
phase_table()
print(f"setup ({SETUP_MODE}): {SETUP_SECONDS:.1f} s")
json.dump({"mode": SETUP_MODE, "setup_seconds": SETUP_SECONDS, "total_seconds": sum(s for _, s in PHASES), "phases": PHASES},
          open(WORK / "phases.json", "w"), indent=1)   # read by CI

## Where to go next

* The real thing: a Mac with Apple Silicon, `shears-cli run` on `google/gemma-4-26B-A4B-it`, with a `--memory-budget`. See the [command line page](https://github.com/a1cd/Shears/blob/main/docs/cli.md) and the [measured results](https://github.com/a1cd/Shears/blob/main/docs/benchmarks.md).
* Turn the export into a GGUF for llama.cpp: `shears-cli gguf --config config.json`, or serve it: `shears-serve serve --model <export dir> --port 8080`.
* What was verified on Linux, and what was not: [docs/linux.md](https://github.com/a1cd/Shears/blob/main/docs/linux.md).

**If the prebuilt binary cannot be set up** the notebook stops in section 2b and says why; the optional last cell builds `shears-cli` from source instead. **If a step of that build fails** the usual suspects are: `swiftly` could not install (try the tarball from swift.org by hand), the build ran out of memory (rerun the cell; MLX's C++ is memory hungry on a 2-core runtime), or a missing system library (the error names it). Open an issue with the output of the failing cell.


## 6. Optional: build `shears-cli` from source (Linux only, off by default)

**You do not need this to run the notebook.** Sections 1 to 5 use the prebuilt binary. This cell is for people changing Shears itself, or on a system the prebuilt binary does not run on (older glibc, not x86_64). It installs the Swift toolchain with `swiftly`, patches a local copy of mlx-swift (two fixes for its Linux CPU build; see [docs/linux.md](https://github.com/a1cd/Shears/blob/main/docs/linux.md)) and builds `shears-cli` from the sources in `SRC`. Expect 15 to 30 minutes on a Colab CPU runtime. On a Mac the notebook already builds from source in section 2a.

To use it: set `BUILD_FROM_SOURCE = True` below, run the cell, and then **rerun from section 3** (the pipeline cells use whatever `SHEARS_CLI` is). If section 2b stopped the notebook, run section 1 and the sources cell in section 2 first.

**EXPERIMENTAL and UNTESTED: CUDA (Linux with an NVIDIA GPU only).**

> **Nobody has run this cell.** It is off by default. Colab's T4 / L4 / A100 images, CUDA 13.0, and the extra headers below have never been tried with Shears, and mlx-swift's own SwiftPM CUDA build is documented as not working on its own: MLX's `.cu` sources include CUTLASS, cuDNN-frontend and CCCL headers that its CMake build downloads and its SwiftPM build does not. This cell downloads those at the versions MLX's CMake build pins and then builds a *second* `shears-cli` from a copy of the sources. If it fails, the compiler error names what is missing; the rest of the notebook still works on the CPU build above. See [docs/linux.md](https://github.com/a1cd/Shears/blob/main/docs/linux.md#read-only-cuda).
>
> To use it set `RUN_EXPERIMENTAL_CUDA = True` in section 1, then run this cell. Needs a GPU runtime with compute capability 7.5 or newer (MLX's stated minimum).

In [ ]:
# Set to True to build from source (see the text above). Default: use the prebuilt binary.
BUILD_FROM_SOURCE = False

if not BUILD_FROM_SOURCE:
    print("skipped: BUILD_FROM_SOURCE is False, the notebook uses the prebuilt shears-cli")
elif not IS_LINUX:
    print("not Linux: on a Mac the notebook builds from source in section 2a")
else:
    # ---- 1. system packages (gfortran: mlx-swift links -lgfortran; g++ stays installed for MLX's run-time kernels) ----
    sh(r"""
SUDO=""; [ "$(id -u)" = 0 ] || SUDO=sudo
export DEBIAN_FRONTEND=noninteractive
. /etc/os-release
$SUDO apt-get update -qq
$SUDO apt-get install -y -qq build-essential git curl ca-certificates python3 pkg-config binutils gnupg2 unzip tzdata \
  zlib1g-dev libncurses-dev libedit2 libsqlite3-0 libcurl4-openssl-dev libxml2-dev libz3-dev \
  libopenblas-dev liblapacke-dev gfortran ninja-build
if [ "$VERSION_ID" = "22.04" ]; then
  # Ubuntu 22.04 ships libstdc++ 12; MLX needs 13 (UNTESTED here). Take it from the toolchain PPA.
  $SUDO apt-get install -y -qq software-properties-common
  $SUDO add-apt-repository -y ppa:ubuntu-toolchain-r/test
  $SUDO apt-get update -qq
fi
$SUDO apt-get install -y -qq libstdc++-13-dev libgcc-13-dev
g++ --version | head -1
""", "apt packages")

    # ---- 2. the Swift toolchain ----
    add_to_path(pathlib.Path.home() / ".local/share/swiftly/bin")
    add_to_path(WORK / "swift-tarball/usr/bin")
    if shutil.which("swift") is None or SWIFT_VERSION not in subprocess.run(["swift", "--version"], capture_output=True, text=True).stdout:
        sh(f"""
# The verified route: swiftly. Run from a scratch directory, because `swiftly install --use` writes a .swift-version file.
mkdir -p "{WORK}/swiftly" && cd "{WORK}/swiftly"
cat > install-swiftly.sh <<'EOF'
curl -sSLO https://download.swift.org/swiftly/linux/swiftly-$(uname -m).tar.gz
tar zxf swiftly-$(uname -m).tar.gz
./swiftly init --quiet-shell-followup --assume-yes --skip-install
. ~/.local/share/swiftly/env.sh
swiftly install --use --assume-yes {SWIFT_VERSION}
EOF
if ! bash -e install-swiftly.sh; then
  echo "swiftly failed; falling back to the toolchain tarball from swift.org (UNTESTED)"
  . /etc/os-release
  P=ubuntu$(echo $VERSION_ID | tr -d .); V=ubuntu$VERSION_ID; X=""
  [ "$(uname -m)" = aarch64 ] && P=$P-aarch64 && X=-aarch64
  curl -fsSL "https://download.swift.org/swift-{SWIFT_VERSION}-release/$P/swift-{SWIFT_VERSION}-RELEASE/swift-{SWIFT_VERSION}-RELEASE-$V$X.tar.gz" -o swift.tar.gz
  mkdir -p "{WORK}/swift-tarball" && tar -xzf swift.tar.gz -C "{WORK}/swift-tarball" --strip-components=1
fi
""", "swift toolchain (swiftly)")
    sh("swift --version", "swift --version")

    # ---- 3. mlx-swift with two local fixes ----
    MLX_SWIFT = WORK / "mlx-swift"
    # The mlx-swift version Shears is pinned to.
    pins = json.load(open(SRC / "Package.resolved"))["pins"]
    mlx_version = next(p["state"]["version"] for p in pins if p["identity"] == "mlx-swift")
    if not (MLX_SWIFT / ".shears-patched").exists():
        sh(f"""
rm -rf "{MLX_SWIFT}"
git clone --quiet --depth 1 --branch {mlx_version} --recurse-submodules --shallow-submodules https://github.com/ml-explore/mlx-swift "{MLX_SWIFT}"
cd "{MLX_SWIFT}"
# 1. MLXNN needs MLXFast, which the Linux CPU manifest leaves out.
git apply "{SRC}/Notebooks/mlx-swift-linux-cpu.patch"
# 2. The shipped compiled_preamble.cpp was generated on arm64 and fails to compile elsewhere; regenerate it here.
cd Source/Cmlx && bash mlx/mlx/backend/cpu/make_compiled_preamble.sh mlx-generated/compiled_preamble.cpp g++ mlx FALSE $(uname -m)
touch "{MLX_SWIFT}/.shears-patched"
""", f"mlx-swift {mlx_version}, patched")
    else:
        print("mlx-swift already patched")

    # ---- 4. shears-cli, CPU ----
    sh(f"""
export SPM_CUDA=0
cd "{SRC}/CLI"
swift package resolve
[ -e Packages/mlx-swift ] || swift package edit mlx-swift --path "{MLX_SWIFT}"
swift build -c release --product shears-cli
""", "swift build shears-cli (release, CPU)")
    SHEARS_CLI = SRC / "CLI/.build/release/shears-cli"

    PREBUILT_OK = False

    # ---- 5. EXPERIMENTAL, UNTESTED: CUDA (RUN_EXPERIMENTAL_CUDA, see above) ----
    if RUN_EXPERIMENTAL_CUDA:
        sh(r"""
    nvidia-smi --query-gpu=name,compute_cap,driver_version --format=csv
    /usr/local/cuda/bin/nvcc --version | tail -2 || { echo 'no CUDA toolkit at /usr/local/cuda: install it first'; exit 1; }
    INC=/usr/local/cuda/include
    T=$(mktemp -d)
    [ -d $INC/cutlass ] || { git clone -q --depth 1 --branch v4.4.2 https://github.com/NVIDIA/cutlass.git $T/cutlass \
        && cp -r $T/cutlass/include/cutlass $T/cutlass/include/cute $INC/; }
    [ -f $INC/cudnn_frontend.h ] || { git clone -q --depth 1 --branch v1.16.0 https://github.com/NVIDIA/cudnn-frontend.git $T/fe \
        && cp -r $T/fe/include/cudnn_frontend $T/fe/include/cudnn_frontend.h $INC/; }
    if [ ! -f $INC/cccl/cuda/std/version ]; then
      curl -fsSL -o $T/cccl.zip https://github.com/NVIDIA/cccl/releases/download/v3.1.3/cccl-v3.1.3.zip
      unzip -q $T/cccl.zip -d $T/cccl && mkdir -p $INC/cccl && { cp -r $T/cccl/*/include/* $INC/cccl/ 2>/dev/null || cp -r $T/cccl/include/* $INC/cccl/; }
    fi
    """, "EXPERIMENTAL cuda headers")
        # A separate copy of the sources, so the CPU build above is left alone (the directory is still called Shears).
        CUDA_SRC = WORK / "cuda" / "Shears"
        shutil.rmtree(CUDA_SRC.parent, ignore_errors=True)
        shutil.copytree(SRC, CUDA_SRC, ignore=shutil.ignore_patterns(".build", "Packages"))
        sh(f"""
    export PATH=/usr/local/cuda/bin:$PATH LD_LIBRARY_PATH=/usr/local/cuda/lib64:${{LD_LIBRARY_PATH:-}}
    export SPM_CUDA=1 CUDA_ARCH=sm_$(nvidia-smi --query-gpu=compute_cap --format=csv,noheader | head -1 | tr -d .)
    cd "{CUDA_SRC}/CLI"
    swift package resolve
    swift package edit mlx-swift --path "{MLX_SWIFT}"
    swift build -c release --product shears-cli
    """, "EXPERIMENTAL swift build (CUDA)")
        SHEARS_CLI = CUDA_SRC / "CLI/.build/release/shears-cli"
        print("using the CUDA build:", SHEARS_CLI)

    print("built from source:", SHEARS_CLI)
